# Energy landscape and correlations

Source: archived main notebook, cells 15–23 and 121–122. Values are uncorrected computed energies.


In [ ]:
from pathlib import Path
import os
import sys

def repository_root(start):
    for path in (start, *start.parents):
        if (path / "Code" / "model_train.py").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the CycloAddGenerator repository.")

REPO_ROOT = repository_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks" / "03_modeling"))
DATA_DIR = REPO_ROOT / "Data" / "DFT_Result"
DESCRIPTOR_MAP = REPO_ROOT / "Data" / "Descriptors" / "Descriptors_.pkl"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from Code import model_train
from _modeling_common import load_modeling_data

FIGURE_DIR = REPO_ROOT / "Figure"
FIGURE_DIR.mkdir(exist_ok=True)
data_Ga, X_Ga, data_G, X_G = load_modeling_data(DATA_DIR, DESCRIPTOR_MAP)
Y_DD = data_Ga["Diene_Distort"].to_numpy()
Y_ED = data_Ga["Ene_Distort"].to_numpy()
Y_IN = data_Ga["Interaction"].to_numpy()
Y_Ga = data_Ga["deltaGa(Solvent)"].to_numpy()
Y_G = data_Ga["deltaG"].to_numpy()

## Descriptor and energy relationships


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

model_train.draw_correlation_map(X_Ga, save_name='Figure/Figure_SN_Descriptors_Correlation', useSVG=1, figure_size=(5,5), annot=False, show_label=True)

In [ ]:
model_train.calc_distribution_line([Y_Ga, Y_G], eachsize=1, ylab=None, return_result = False, colors=['#df851a', '#7d9263'], save_name='Figure/Figure_3_FreeEnergies_Distribution', useSVG=1, figure_size=(8,4.5))
# Y_G, '#7d9263'

In [ ]:
model_train.calc_distribution_line([Y_DD, Y_ED, -1 * Y_IN], eachsize=1, ylab=None, return_result = False, colors=['#b55475', '#7153a1', '#4e75a3'], save_name='Figure/Figure_3_DIEnergies_Distribution', useSVG=1, figure_size=(8,4.5))

In [ ]:
model_train.calc_distribution_line([Y_Ga, Y_G, Y_DD, Y_ED, -1 * Y_IN], eachsize=1, ylab=None, return_result = False, colors=['#df851a', '#7d9263', '#b55475', '#7153a1', '#4e75a3'], save_name='Figure/Figure_3_Energies_Distribution', useSVG=1, figure_size=(15,6))

In [ ]:
model_train.plot_scatter_with_metrics(Y_G, Y_Ga, alpha=0.5, prediction=False, cmap="Blues", scatter=True, xticks=range(-60, 10, 20))
linear = LinearRegression()
linear.fit(Y_G.reshape(-1, 1), Y_Ga)
R2 = linear.score(Y_G.reshape(-1, 1), Y_Ga)
print("R2:", R2)
from sklearn.metrics import mean_squared_error, mean_absolute_error
mse = mean_squared_error(Y_G, Y_Ga)
mae = mean_absolute_error(Y_G, Y_Ga)
print("MSE:", mse)
print("MAE:", mae)
pearson_r = np.corrcoef(Y_G, Y_Ga)[0, 1]
print("Pearson R:", pearson_r)

In [ ]:
data = Y_DD
counts, bin_edges = np.histogram(data, bins=30)
peak_idx = np.argmax(counts)
peak_range = (bin_edges[peak_idx], bin_edges[peak_idx + 1])
peak_center = (peak_range[0] + peak_range[1]) / 2
print("4π Distortion Perk:", peak_center)
data = Y_ED
counts, bin_edges = np.histogram(data, bins=30)
peak_idx = np.argmax(counts)
peak_range = (bin_edges[peak_idx], bin_edges[peak_idx + 1])
peak_center = (peak_range[0] + peak_range[1]) / 2
print("2π Distortion Perk:", peak_center)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

data = np.array([Y_Ga, Y_G, Y_DD, Y_ED, -1 * Y_IN, Y_DD / (Y_DD + Y_ED), Y_DD + Y_ED]).T
# model_train.draw_correlation_map(data, save_name='Figure/Figure_4_Energies_Correlation', useSVG=1, figure_size=(5,5))
df = pd.DataFrame(data)
# correlation_matrix = np.abs(df.corr())

n = len(df.columns)
correlation_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        if i == j:
            # Set the diagonal to one
            correlation_matrix[i, j] = 1.0
        else:
            # Regress column i on column j
            X = df.iloc[:, j].values.reshape(-1, 1)
            y = df.iloc[:, i].values
            model = LinearRegression()
            model.fit(X, y)
            correlation_matrix[i, j] = model.score(X, y)

mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
print(np.max(correlation_matrix[~mask]))
f, ax = plt.subplots(figsize=(5, 4), dpi=300)
annot_kws = {"fontsize": 12}
ax = sns.heatmap(correlation_matrix, 
        mask=mask,
        cmap='coolwarm',    
        annot=True,         
        fmt='.2f',          
        center=0,           
        cbar=1,
        annot_kws=annot_kws,)
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=18)
# if not show_label:
ax.set_xticklabels([])
ax.set_yticklabels([])
# plt.tight_layout()
# if useSVG:
#     plt.savefig(f"{save_name}.svg", format="svg", bbox_inches='tight')
# else:
plt.savefig(str(FIGURE_DIR / "Figure_3_Correlation.png"), dpi=300, bbox_inches='tight')

In [ ]:
model_train.calc_distribution_line([Y_IN / (Y_DD + Y_ED)], eachsize=0.01, ylab=None, return_result = False, colors=['#df851a', '#7d9263', '#b55475', '#7153a1', '#4e75a3'], save_name='Figure/4pd_alld', useSVG=1, figure_size=(6,5), xlimit=[0,1])

## Barrier-window figures


In [ ]:
# Separate low-height D/I distributions for each narrow aqueous-barrier window
from scipy.ndimage import gaussian_filter1d
from pathlib import Path
import matplotlib.pyplot as plt

FIGURE_DIR.mkdir(exist_ok=True)
barrier = np.asarray(Y_Ga, dtype=float)
total_dist = np.asarray(Y_DD + Y_ED, dtype=float)
interaction = np.asarray(Y_IN, dtype=float)
windows = [(lo, lo + 1) for lo in range(10, 61, 5)]

fig, axes = plt.subplots(len(windows), 2, figsize=(12, 13.5), dpi=300, sharey="col")
for col, (values, xlabel, color) in enumerate([
    (total_dist, "Total distortion (kcal mol$^{-1}$)", "#b55475"),
    (interaction, "Interaction stabilization (kcal mol$^{-1}$)", "#4e75a3"),
]):
    bins = np.arange(np.floor(values.min()) - 1, np.ceil(values.max()) + 1.5, 1.0)
    centers = (bins[:-1] + bins[1:]) / 2
    ymax = 0
    for row, (lo, hi) in enumerate(windows):
        ax = axes[row, col]
        subset = values[(barrier >= lo) & (barrier < hi)]
        counts, _ = np.histogram(subset, bins=bins)
        density = counts.astype(float) / max(counts.sum(), 1)
        smooth = gaussian_filter1d(density, sigma=0.8) if len(density) >= 3 else density
        line_x = np.linspace(centers[0], centers[-1], 500)
        line_y = np.interp(line_x, centers, smooth)
        ymax = max(ymax, float(line_y.max()))
        ax.plot(line_x, line_y, color=color, lw=1.5)
        ax.fill_between(line_x, line_y, 0, color=color, alpha=0.18)
        ax.text(0.01, 0.80, f"{lo}\u2013{hi} (n={len(subset)})", transform=ax.transAxes, fontsize=8, color=color)
        ax.grid(axis="y", alpha=0.20)
        ax.tick_params(axis="both", labelsize=7, length=2)
        if row < len(windows) - 1: ax.tick_params(labelbottom=False)
        if col == 1: ax.tick_params(labelleft=False)
        else: ax.set_ylabel("freq.", fontsize=8)
        if row == 0: ax.set_title(xlabel, fontsize=11, pad=18)
        if row == len(windows) - 1: ax.set_xlabel("Energy (kcal mol$^{-1}$)", fontsize=9)
    for row in range(len(windows)):
        axes[row, col].set_ylim(0, ymax * 1.15 if ymax else 1)
fig.subplots_adjust(left=0.07, right=0.98, top=0.965, bottom=0.035, hspace=0.12, wspace=0.08)
fig.savefig(str(FIGURE_DIR / "Figure_BarrierWindow_Distortion_Interaction_Overlay.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Horizontal ridge distributions across contiguous 5 kcal/mol aqueous-barrier windows
from scipy.ndimage import gaussian_filter1d
from pathlib import Path
import matplotlib.pyplot as plt

FIGURE_DIR.mkdir(exist_ok=True)
barrier = np.asarray(Y_Ga, dtype=float)
total_dist = np.asarray(Y_DD + Y_ED, dtype=float)
interaction = np.asarray(-Y_IN, dtype=float)
windows_5kcal = [(lo, lo + 1) for lo in range(25, 51, 5)]
window_labels = [f"{lo}\u2013{hi}" for lo, hi in windows_5kcal]

plot_specs = [
    (total_dist, "Total distortion", "Total distortion energy (kcal mol$^{-1}$)", "#b55475", "Distortion"),
    (interaction, "Interaction", "Interaction energy (kcal mol$^{-1}$)", "#4e75a3", "Interaction"),
]
for values, title, xlabel, color, file_label in plot_specs:
    fig, ax = plt.subplots(figsize=(4.5, 4.5), dpi=300, constrained_layout=True)
    valid_values = values[np.isfinite(values)]
    energy_bins = np.arange(np.floor(valid_values.min()) - 1, np.ceil(valid_values.max()) + 1.5, 1.0)
    energy_centers = (energy_bins[:-1] + energy_bins[1:]) / 2
    energy_grid = np.linspace(energy_centers[0], energy_centers[-1], 600)

    curves, sample_sizes = [], []
    for lo, hi in windows_5kcal:
        mask = (barrier >= lo) & (barrier < hi) & np.isfinite(barrier) & np.isfinite(values)
        subset = values[mask]
        counts, _ = np.histogram(subset, bins=energy_bins)
        density = counts.astype(float) / max(counts.sum(), 1)
        smooth = gaussian_filter1d(density, sigma=0.8) if density.size >= 3 else density
        curves.append(np.interp(energy_grid, energy_centers, smooth))
        sample_sizes.append(subset.size)

    panel_peak = max((curve.max() for curve in curves), default=0.0)
    panel_peak = panel_peak if panel_peak > 0 else 1.0
    for slot, (curve, n_sample) in enumerate(zip(curves, sample_sizes)):
        baseline = float(slot)
        ridge_y = baseline + 0.82 * curve / panel_peak
        ax.fill_between(energy_grid, baseline, ridge_y, color=color, alpha=0.24, linewidth=0)
        ax.plot(energy_grid, ridge_y, color=color, lw=1.35)
        ax.axhline(baseline, color="0.78", lw=0.65, zorder=0)
        # ax.text(1.005, baseline + 0.41, f"n={n_sample}", transform=ax.get_yaxis_transform(),
        #         ha="left", va="center", fontsize=20, color="0.35", clip_on=False)

    # ax.set_xlim(0, energy_grid[-1])
    # ax.set_ylim(-0.08, len(windows_5kcal) - 0.08)
    # ax.set_yticks(np.arange(len(windows_5kcal)) + 0.41, labels=window_labels)
    # ax.set_xticks(np.arange(0, energy_grid[-1] + 1, 20 if title == "Total distortion" else 10))

    ax.set_xlim(0, energy_grid[-1])
    ax.set_xticks(np.arange(0, energy_grid[-1] + 1, 20))
    if title == "Interaction":
        ax.set_xlim(energy_grid[0], 0)
        ax.set_xticks(np.arange(-40, 10, 20))
    ax.set_ylim(-0.08, len(windows_5kcal) - 0.08)
    ax.set_yticks(np.arange(len(windows_5kcal)) + 0.41, labels=window_labels)
    # ax.set_xlabel(xlabel, fontsize=10)
    # ax.set_ylabel("Aqueous barrier window, $\\Delta G^{\\ddagger}_{\\mathrm{solv}}$ (kcal mol$^{-1}$)", fontsize=20)
    # ax.set_title(title, loc="left", fontsize=24, color=color, pad=16)
    ax.grid(axis="x", alpha=0.18, linewidth=0.6)
    ax.tick_params(axis="x", labelsize=24)
    ax.tick_params(axis="y", labelsize=24, length=0, pad=5)
    ax.spines[["top", "right"]].set_visible(False)
    output_stem = str(FIGURE_DIR / f"Figure_BarrierWindow_{file_label}_HorizontalRidges")
    fig.savefig(f"{output_stem}.png", dpi=300, bbox_inches="tight")
    fig.savefig(f"{output_stem}.svg", bbox_inches="tight")
    plt.show()